# Worksheet: Torch

In this worksheet, we will use Torch to train a general linear regression model. Compared with Worksheet 10, we will have more than 2 trainable parameters and matrix vector multiplicaton is used.

The following code generates training and test samples

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt


datadim = 5           # intrinsic dimension of data
dim = 500             # dimension of the ambient space
N = 5000              # data set size
eps = 0.5             # noise level in the observed data

############# training samples
# create a low-dimensional data set
Xreal = torch.normal(mean =0, std=1.0, size=[N,datadim])

# compute the target function
Y = torch.tanh(Xreal[:,0])

# a matrix to embed the given data into a high-dimensional ambient space
transform = torch.normal(mean =0, std=1.0, size=[datadim,dim])

# the high-dimensional data is obtained by matrix multiplication
X = torch.matmul(Xreal, transform)

# making the observations noisy
Y += torch.normal (mean = 0, std=eps, size = Y.shape )

# Note: If you generate a dataset in numpy and then convert it to torch tensors, you may run into trouble with default data types.
# The default for torch is float32.

############# test samples
N_test = 10000

# create a low-dimensional data set
Xreal = torch.normal(mean = 0, std = 1.0, size=[N_test,datadim])

# compute the target function
Y_test = torch.tanh(Xreal[:,0])

# a matrix to embed the given data into a high-dimensional ambient space
transform = torch.normal(mean = 0, std=1.0, size=[datadim,dim])

# the high-dimensional data is obtained by matrix multiplication
X_test = torch.matmul(Xreal, transform)

In [ ]:
# defining the function for forward pass for prediction
def forward(x):
    return x @ w + b
 
# evaluating data points with Mean Square Error.
def criterion(y_pred, y):
    return torch.mean((y_pred - y) ** 2)
 
w = torch.zeros([dim,1], requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
 
step_size = 0.01
loss_list = []
iter = 500

loss_test_list = []

for i in range (iter):    
    # making predictions with forward pass (training)
    Y_pred = forward(X)
    
    # making predictions on test data
    y_pred = forward(X_test)
    loss_test = criterion(y_pred, Y_test)
    loss_test_list.append(loss_test.item())
    
    # calculating the loss between original and predicted data points
    loss = criterion(Y_pred, Y)
    # storing the calculated loss in a list
    loss_list.append(loss.item())
    # backward pass for computing the gradients of the loss w.r.t to learnable parameters
    loss.backward()
    # updateing the parameters after each iteration
    w.data = w.data - step_size * w.grad.data
    b.data = b.data - step_size * b.grad.data
    # zeroing gradients after each iteration
    w.grad.data.zero_()
    b.grad.data.zero_()
    
    if i % 100 == 0:
        # priting the values for understanding
        print('{}, \t{}'.format(i, loss.item()))
    
    
    
# visualization
plt.plot(loss_list, 'r', label='training loss')
plt.plot(loss_test_list, 'b', label='test loss')
plt.legend()
#plt.tight_layout()
#plt.grid('True', color='y')
plt.xlabel("Iterations")
plt.ylabel("Loss")
#plt.show()
#plt.savefig("torch.jpg")